# Paper figures (matplotlib)

Draws Figs. 1–8 of the manuscript from `data/figure_data.jld2`, which the export cell of `make_figures.ipynb`
(Julia) writes; its last cell then appends the square-well saturation pressures. All physics stays in Julia; this notebook only plots.

Style: SciencePlots `science` with APS sizes. Each figure is created at its printed width
(single column 3.4 in, double column 7.0 in), so the font sizes below are the sizes in print.
Python packages: `requirements.txt`.

In [ ]:
import numpy as np, h5py, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import scienceplots

plt.style.use(["science"])
COLW, TEXTW = 246 / 72, 510 / 72          # revtex4-2 column and text widths (in)
mpl.rcParams.update({
    "font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "axes.titlesize": 9,
    "axes.linewidth": 0.8, "lines.linewidth": 1.3, "lines.markersize": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "xtick.minor.width": 0.6, "ytick.minor.width": 0.6,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5, "xtick.minor.size": 2, "ytick.minor.size": 2,
    "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.3", "legend.fancybox": False,
    "legend.handlelength": 1.6, "legend.borderpad": 0.35, "legend.labelspacing": 0.25,
    "savefig.bbox": "standard", "savefig.dpi": 600, "figure.constrained_layout.use": True,
    "figure.constrained_layout.h_pad": 0.02, "figure.constrained_layout.w_pad": 0.02,
})

# Colour scheme: five hues, one role each, across all figures —
#   BLUE this work, 1st order, SW λ=1.50 | RED T_c estimate | MAGENTA 3rd order | PURPLE the ring
#   ORANGE outside the ring, 2nd order, quantities from the zeros, SW λ=1.75 | TEAL inside the ring
# Reference data (ln Q, computed zeros, experiment, TMMC) black/grey. Temperature: one cold → hot ramp
# (navy → violet → crimson), never fading to white, also used for the T/T_c colour bar.
BLUE, RED, PURPLE, ORANGE, TEAL = "#1565c0", "#d32f2f", "#7b1fa2", "#ef6c00", "#00897b"
MAGENTA = "#ad1457"   # 3rd order: red was too close to orange (2nd order) for colour-blind readers
# Comparison figures (Figs. 1c, 4c, 5, 6): BLUE (same #1565c0 throughout) = this work in Figs. 1c and 6,
# SW λ=1.75, run 2; GREEN = SW λ=1.50, run 1; VIOLET = run 3; CHARCOAL = run 4. Reference data stay black.
GREEN, VIOLET, CHARCOAL = "#2f8f46", "#9c6ade", "#3c3c3c"
REF, INK2 = "#111111", "#52514e"
TGRAD = LinearSegmentedColormap.from_list("coldhot", ["#1f3a93", "#7b3fa0", "#c0263e"])
COLC = {0: TEAL, 1: PURPLE, 2: ORANGE}; MKC = {0: "^", 1: "o", 2: "D"}
CLASS_LABEL = {0: "Inside the ring", 1: "On the ring", 2: "Outside the ring"}

D = h5py.File("data/figure_data.jld2", "r")
g = lambda k: D[k][()]
cz = lambda k: D[k + "_re"][()] + 1j * D[k + "_im"][()]
TEMPS = [t.decode() if isinstance(t, bytes) else t for t in g("temps")]
tcol = lambda Ts: TGRAD(np.clip((Ts - 0.75) / 0.45, 0, 1))

# Panel label (a), (b), … inside the frame, top left (or top right where the data sit top left); x overrides;
# extra is appended after the letter, e.g. the state point.
def panel(ax, tag, right=False, x=None, extra=""):
    ax.text(x if x is not None else (0.965 if right else 0.035), 0.965, f"({tag})" + (f" {extra}" if extra else ""), transform=ax.transAxes, ha="right" if right else "left",
            va="top", fontsize=10, zorder=10,
            bbox=dict(fc="white", ec="none", pad=0.6))

LABEL_GAP_PT = 1.5   # gap between an axis label and the tick numbers beside it

def _shift_label(ax, axis, lab, dx_px, dy_px):
    "Move an axis label by (dx, dy) pixels, keeping it in axes coordinates thereafter."
    anchor = lab.get_transform().transform(lab.get_position())
    xa, ya = ax.transAxes.inverted().transform((anchor[0] + dx_px, anchor[1] + dy_px))
    axis.set_label_coords(xa, ya)

# Place each axis label LABEL_GAP_PT from the tick numbers that actually sit beside it (not from the widest
# number on the axis), panel by panel, so labels hug their own axes. Iterated because constrained layout
# re-flows the panels once the labels move.
def tuck_labels(fig, iters=4):
    gap = LABEL_GAP_PT * fig.dpi / 72
    for _ in range(iters):
        fig.canvas.draw(); r = fig.canvas.get_renderer()
        for ax in fig.axes:
            if ax.get_label() == "<colorbar>":
                continue
            axbb = ax.get_window_extent(r)
            lab = ax.yaxis.label                      # y: right edge just left of the numbers beside it
            if lab.get_text() and ax.yaxis.get_label_position() == "left":
                lb = lab.get_window_extent(r)
                near = [b for b in (t.get_window_extent(r) for t in ax.get_yticklabels() if t.get_visible() and t.get_text())
                        if b.y1 > lb.y0 and b.y0 < lb.y1]
                left = min(b.x0 for b in near) if near else axbb.x0
                _shift_label(ax, ax.yaxis, lab, (left - gap) - lb.x1, 0)
            lab = ax.xaxis.label                      # x: top edge just below the numbers above it
            if lab.get_text() and ax.get_xticklabels() and any(t.get_text() for t in ax.get_xticklabels()):
                lb = lab.get_window_extent(r)
                near = [b for b in (t.get_window_extent(r) for t in ax.get_xticklabels() if t.get_visible() and t.get_text())
                        if b.x1 > lb.x0 and b.x0 < lb.x1]
                bottom = min(b.y0 for b in near) if near else axbb.y0
                _shift_label(ax, ax.xaxis, lab, 0, (bottom - gap) - lb.y1)

def save(fig, name):
    # tick numbers a little closer to the axes; each axis label then tucked against its own tick numbers
    for ax in fig.axes:
        ax.tick_params(axis="both", which="major", pad=2)
    tuck_labels(fig)
    # titled legends (the T* color keys of Figs. 1a, 2, 4a,b): title centered over the entries
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is not None and leg.get_title().get_text():
            leg.set_alignment("center"); leg.get_title().set_multialignment("center")
    for ext in ("pdf", "png"):
        fig.savefig(f"{name}.{ext}")
    print(f"wrote figures/{name}.pdf")

## Fig. 1 — validation, LJ fluid (text width)

In [ ]:
fig, (a, b, c) = plt.subplots(1, 3, figsize=(TEXTW, 2.3))
for T in TEMPS:
    k = f"fig1/T{T}/"; Ts = g(k + "Tstar")
    a.plot(g(k + "N"), g(k + "lnQ"), color=tcol(Ts), lw=1.0, label=f"{Ts:.2f}")
    b.plot(g(k + "N"), g(k + "PN"), color=tcol(Ts), lw=1.0)
lnQmax = max(g(f"fig1/T{T}/lnQ").max() for T in TEMPS)
a.set(xlabel=r"$N$", ylabel=r"$\ln Q(N,V,T)$", xlim=(0, 475), ylim=(0, 1.04 * lnQmax))   # origin at the corner; room to see where the lines end
a.legend(title=r"$T^*$", ncol=2, loc="lower right", bbox_to_anchor=(1.018, 0),   # nudged right off the curves   # below the curves, which rise to the upper right
         fontsize=7.2, title_fontsize=7.7, columnspacing=0.6, handlelength=1.0, handletextpad=0.4, labelspacing=0.15,
         borderaxespad=0.9)   # a little white space to the right and below; top-left corner just clear of the curves
# x ends where the rightmost (T* = 0.75) liquid peak ends; y just above the tallest (T* = 0.75) gas peak
Pmax = max(g(f"fig1/T{T}/PN").max() for T in TEMPS)
b.set(xlabel=r"$N$", ylabel=r"$P(N)$ at $z_c$", xlim=(0, 440), ylim=(0, 1.02 * Pmax))
TK = g("fig1/T_K")
# experiment as a line; TMMC as open circles around our filled points
c.plot(g("fig1/exp_liq"), TK, "-", color=REF, lw=1.8, label="Experiment")
c.plot(g("fig1/exp_vap"), TK, "-", color=REF, lw=1.8)
c.plot(g("fig1/tmmc_liq"), TK, "o", mfc="none", mec="black", ms=6.5, mew=0.8, label="TMMC", zorder=3)
c.plot(g("fig1/tmmc_vap"), TK, "o", mfc="none", mec="black", ms=6.5, mew=0.8, zorder=3)
c.errorbar(g("fig1/rho_liq"), TK, xerr=g("fig1/rho_liq_sd"), fmt="o", color=BLUE, ms=3, label="This work", zorder=4)
c.errorbar(g("fig1/rho_vap"), TK, xerr=g("fig1/rho_vap_sd"), fmt="o", color=BLUE, ms=3, zorder=4)
c.plot(g("fig1/rhoc"), g("fig1/Tc"), "*", color=BLUE, ms=8, label=r"$T_c$, this work")
c.plot(g("fig1/exp_rhoc"), g("fig1/exp_Tc"), "*", mfc="white", mec=REF, ms=8, mew=0.8, label=r"$T_c$, experiment")
c.set(xlabel=r"$\rho$ (g/cm$^3$)", ylabel=r"$T$ (K)", xlim=(-0.07, 1.5))   # white sliver left of the vapor TMMC circles
h, l = c.get_legend_handles_labels(); order = [l.index(x) for x in ("This work", "TMMC", "Experiment",
                                                                     r"$T_c$, this work", r"$T_c$, experiment")]
c.legend([h[i] for i in order], [l[i] for i in order], loc="lower center", bbox_to_anchor=(0.46, 0.02), fontsize=7,
         handlelength=1.3)
fig.get_layout_engine().set(wspace=0.12)             # more room between panels (Fig. 1 only)
panel(a, "a"); panel(b, "b", x=0.07); panel(c, "c")   # (b) clear of the tallest gas peak at N ≈ 1
save(fig, "fig1_validation")

## Figs. 2 and 3 — LJ zero maps

In [ ]:
SYS = [(T, g(f"lj/T{T}/Tstar"), cz(f"lj/T{T}/z"), g(f"lj/T{T}/zc"), g(f"lj/T{T}/cls")) for T in TEMPS]
zmax = max(s[3] for s in SYS)

# Fig. 2 (column width): every zero at all ten isotherms, axes in units of 1e-5
fig, ax = plt.subplots(figsize=(COLW, 2.7))
ZU = 1e-5
for T, Ts, z, zc, cls in SYS:
    ax.scatter(z.real / ZU, z.imag / ZU, s=2.2, color=tcol(Ts), lw=0, label=f"{Ts:.2f}")
L = 1.3 * zmax / ZU
ax.set(xlim=(-L, L), ylim=(-L, L), aspect="equal",
       xlabel=r"$\mathrm{Re}\,z\;(10^{-5})$", ylabel=r"$\mathrm{Im}\,z\;(10^{-5})$")
ax.legend(title=r"$T^*$", loc="upper left", bbox_to_anchor=(1.02, 1.0), markerscale=3, fontsize=7.5,
          title_fontsize=8, handletextpad=0.2, borderaxespad=0)
save(fig, "fig2_all_zeros")

# Fig. 3 (column width, two rows): (a) zeros by class at each isotherm; (b), (c) maps at T* = 0.75 and 1.20 below
fig, axd = plt.subplot_mosaic([["a", "a"], ["b", "c"]], figsize=(COLW, 3.7), gridspec_kw={"height_ratios": [1, 1.05]})
a, b, c = axd["a"], axd["b"], axd["c"]
Ts_ = np.array([s[1] for s in SYS])
n = {k: np.array([np.sum(s[4] == k) for s in SYS]) for k in (0, 1, 2)}
a.bar(Ts_, n[1], width=0.035, color=COLC[1], label=CLASS_LABEL[1])
a.bar(Ts_, n[0], width=0.035, bottom=n[1], color=COLC[0], label=CLASS_LABEL[0])
a.bar(Ts_, n[2], width=0.035, bottom=n[1] + n[0], color=COLC[2], label=CLASS_LABEL[2])
a.set(xlabel=r"$T^*$", ylabel="Number of zeros", ylim=(0, 720), xticks=[0.8, 0.9, 1.0, 1.1, 1.2])
h, l = a.get_legend_handles_labels(); a.legend(h[::-1], l[::-1], loc="upper center", ncol=1, borderaxespad=0.25)   # slivers above the bars and below the top axis
a.minorticks_off(); a.tick_params(axis="x", which="both", top=False)

def map_panel(ax, s, lim=2.0):
    T, Ts, z, zc, cls = s; w = z / zc
    phi = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(phi), np.sin(phi), color=INK2, ls="--", lw=0.6)
    for k in (2, 0, 1):
        sel = cls == k
        ax.scatter(w[sel].real, w[sel].imag, s=10 if k != 1 else 5, marker=MKC[k], color=COLC[k], lw=0)
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), aspect="equal", xlabel=r"$\mathrm{Re}\,z/z_c$",
           ylabel=r"$\mathrm{Im}\,z/z_c$", xticks=[-2, -1, 0, 1, 2], yticks=[-2, -1, 0, 1, 2])
byT = {s[0]: s for s in SYS}
map_panel(b, byT["87.79"]); map_panel(c, byT["140.46"]); c.set(ylabel=""); c.tick_params(labelleft=False)   # (c) shares the y axis of (b)
panel(a, "a"); panel(b, "b", extra=r"$T^*=0.75$"); panel(c, "c", extra=r"$T^*=1.20$")   # may sit over a stray zero
save(fig, "fig3_zero_classes")

## Fig. 4 — SW fluid (text width)

In [ ]:
fig, (a, b, c) = plt.subplots(1, 3, figsize=(TEXTW, 2.35), gridspec_kw={"width_ratios": [1.15, 1.15, 1]})
SWCOL = {"1.50": GREEN, "1.75": BLUE}
def sw_keys(lam): return sorted(D[f"sw/lam{lam}"].keys(), key=lambda k: float(k[1:]))
for ax, lam in ((a, "1.50"), (b, "1.75")):
    ks = sw_keys(lam); Ts = [g(f"sw/lam{lam}/{k}/Tstar") for k in ks]
    zm = max(np.exp(g(f"sw/lam{lam}/{k}/lnzc")) for k in ks)
    for k, T in zip(ks, Ts):
        z = cz(f"sw/lam{lam}/{k}/z")
        ax.scatter(z.real, z.imag, s=2.2, lw=0, color=TGRAD((T - Ts[0]) / (Ts[-1] - Ts[0])), label=f"{T:.2f}")
    L = 1.3 * zm
    ax.set(xlim=(-L, L), ylim=(-L, L), aspect="equal", xlabel=r"$\mathrm{Re}\,z$", ylabel=r"$\mathrm{Im}\,z$",
           xticks=[-0.04, 0, 0.04], yticks=[-0.04, 0, 0.04])
    ax.legend(title=rf"$\lambda={lam}$" + "\n" + r"$T^*$", loc="upper left", bbox_to_anchor=(1.02, 1.0), markerscale=3, fontsize=7.5,
              title_fontsize=8, handletextpad=0.2, borderaxespad=0)
# (c) saturation pressure, a visual check; the ln z_c agreement with NIST is stated in the text.
# References: NIST SAT-TMMC for λ = 1.50 in the same box, Singh, Kofke & Errington (2003) Table IV for λ = 1.75.
c.plot(g("sw/nist_P_T"), g("sw/nist_P"), "s", mfc="none", mec=INK2, ms=5.5, mew=0.8, label="NIST", zorder=3)
c.plot(g("sw/singh175_T"), g("sw/singh175_P"), "o", mfc="none", mec=INK2, ms=5.5, mew=0.8, label=r"Singh \textit{et al.}", zorder=3)
for lam in ("1.50", "1.75"):
    ks = sw_keys(lam)
    c.errorbar([g(f"sw/lam{lam}/{k}/Tstar") for k in ks], [g(f"sw/lam{lam}/{k}/Psat") for k in ks],
               yerr=[g(f"sw/lam{lam}/{k}/Psat_sd") for k in ks], fmt="-", marker={"1.50": "D", "1.75": "^"}[lam], color=SWCOL[lam], ms=3.5,   # shapes as in Fig. 8(a)
               label=rf"$\lambda={lam}$")
c.set(xlabel=r"$T^*$", ylabel=r"$P^*_{\mathrm{sat}}$", xticks=[1.0, 1.2, 1.4, 1.6],
      xlim=(0.86, 1.77), ylim=(0.010, 0.091))   # white space around the end points
c.legend(loc="upper left", bbox_to_anchor=(0.215, 0.985), fontsize=6.5, handlelength=1.0, borderaxespad=0.3)   # top strip between the two curves
for ax, t in zip((a, b, c), "abc"): panel(ax, t)
save(fig, "fig4_square_well")

## Fig. 5 — independent runs (column width)

In [ ]:
fig, ax = plt.subplots(figsize=(COLW, 3.36))   # height at which the square axes fill the column width (centred)
w = cz("fig5/avg")
# each run its own colour and marker; the averaged-ln Q zeros as open black circles on top (as computed zeros, Fig. 7), with a
# bold edge so they read through the dense ring and the filled run markers show inside them
# drawn in reverse and slightly transparent, so no single run covers the ring; legend kept in run order
for r, col, m in reversed(list(zip(range(1, 5), (GREEN, BLUE, VIOLET, CHARCOAL), ("o", "s", "^", "D")))):
    x = cz(f"fig5/run{r}"); ax.scatter(x.real, x.imag, s=10, marker=m, color=col, lw=0, alpha=0.75, label=f"Run {r}")
ax.scatter(w.real, w.imag, s=30, facecolor="none", edgecolor=REF, lw=1.0, zorder=5, label=r"Averaged $\ln Q$")   # larger than the run markers (s=10)
ax.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), aspect="equal", xlabel=r"$\mathrm{Re}\,z/z_c$", ylabel=r"$\mathrm{Im}\,z/z_c$")
# inset, lower right (outside the ring, clear of the legend): the first 10 ring zeros above the positive real axis,
# where neighbouring zeros are resolved and the run markers sit inside the averaged-ln Q circles
axi = ax.inset_axes([0.69, 0.03, 0.28, 0.28])
for r, col, m in reversed(list(zip(range(1, 5), (GREEN, BLUE, VIOLET, CHARCOAL), ("o", "s", "^", "D")))):
    x = cz(f"fig5/run{r}"); axi.scatter(x.real, x.imag, s=10, marker=m, color=col, lw=0, alpha=0.75)
axi.scatter(w.real, w.imag, s=30, facecolor="none", edgecolor=REF, lw=1.0, zorder=5)
ring = w[(abs(np.log(abs(w))) < 0.03) & (w.imag > 0)]; ring = ring[np.argsort(np.angle(ring))][:11]
edge = (ring[9] + ring[10]) / 2   # frame halfway between the 10th and 11th zeros, so no zero is cut by it
axi.set(xlim=(edge.real, 1 + 0.25 * (1 - edge.real)), ylim=(0, edge.imag))   # bottom edge = real axis, as Fig. 7(d-f)
axi.set(xticks=[], yticks=[])   # no tick labels: they would sit on the zeros of the main panel; range given in the caption
ax.indicate_inset_zoom(axi, edgecolor=INK2, lw=0.6, alpha=1)
h, l = ax.get_legend_handles_labels(); o = sorted(range(len(l)), key=lambda i: (not l[i].startswith("Run"), l[i]))
ax.legend([h[i] for i in o], [l[i] for i in o], loc="upper right", markerscale=1.5)
save(fig, "fig5_independent_runs")

## Fig. 6 — thermodynamics from the zeros (column width, panels stacked on a shared $\beta\mu$ axis)

In [ ]:
fig, (a, b) = plt.subplots(2, 1, figsize=(COLW, 3.6), sharex=True)
a.plot(g("fig6/t"), g("fig6/rho"), color=REF, lw=1.0, alpha=0.75, label=r"From $\ln Q$")
a.plot(g("fig6/tm"), g("fig6/rho_roots"), "o", color=BLUE, ms=4, mew=0, alpha=0.65, label="From the zeros")
a.set(ylabel=r"$\rho\;(\sigma^{-3})$"); a.legend(loc="lower right")
b.semilogy(g("fig6/t"), g("fig6/kappa"), color=REF, lw=1.0, alpha=0.75, label=r"From $\ln Q$")
b.semilogy(g("fig6/tm"), g("fig6/kappa_roots"), "o", color=BLUE, ms=4, mew=0, alpha=0.65, label="From the zeros")
b.set(xlabel=r"$\beta\mu-\beta\mu_c$", ylabel=r"$\kappa_T\;(\sigma^3/\epsilon)$")   # legend in (a) only
for ax, t in zip((a, b), "ab"): panel(ax, t)
save(fig, "fig6_thermo_from_roots")

## Fig. 7 — Lee's expansion against the zeros (text width)

In [ ]:
ORD = ((1, BLUE, "D", r"1st order ($\Delta_1$)"), (2, ORANGE, "^", r"+ 2nd order ($\Delta_2$)"),
       (3, MAGENTA, "*", r"+ 3rd order ($\Delta_3$)"))
fig, axs = plt.subplots(2, 3, figsize=(TEXTW, 4.6))
for j, T in enumerate(("87.79", "134.61", "140.46")):
    k = f"fig7/T{T}/"; w = cz(k + "w"); ax = axs[0, j]
    ax.scatter(w.real, w.imag, s=7, facecolor="none", edgecolor="black", lw=0.35, alpha=0.75, label="Computed zeros", zorder=6)   # open, on top: predictions show through
    for n, col, m, lab in ORD:
        z = cz(k + f"full/z{n}"); z = np.concatenate([z, z.conj()]); z = z[np.isfinite(z) & (abs(z) < 3)]
        ax.scatter(z.real, z.imag, s=14 if m != "*" else 22, marker=m, color=col, lw=0, alpha=0.65, label=lab, zorder=5 - n)   # 1st over 2nd over 3rd; open computed zeros on top   # large enough to tell the shapes apart
    ax.set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5), aspect="equal", xticks=[-1, 0, 1], yticks=[-1, 0, 1],
           xlabel=r"$\mathrm{Re}\,z/z_c$", ylabel=r"$\mathrm{Im}\,z/z_c$")
    if j == 0: ax.legend(loc="lower left", markerscale=1.3, fontsize=7).set_zorder(20)
    # zoom on the first 25 zeros above the positive real axis
    ax = axs[1, j]; zp = [cz(k + f"zoom/z{n}") for n in (1, 2, 3)]
    pts = np.concatenate([cz(k + "zoom/exact")] + zp)
    xr = (pts.real.min(), pts.real.max()); yr = (0.0, pts.imag.max()); pad = 0.06 * (yr[1] - yr[0])
    ax.scatter(w.real, w.imag, s=34, facecolor="none", edgecolor="black", lw=0.8, alpha=0.75, zorder=6)   # open, on top
    for (n, col, m, lab), z in zip(ORD, zp):
        ax.scatter(z.real, z.imag, s=13 if m != "*" else 24, marker=m, color=col, lw=0, alpha=0.65,
                   zorder=5 - n)   # 1st over 2nd over 3rd
    ax.set(xlim=(xr[0] - pad, xr[1] + pad), ylim=(0, yr[1] + pad),   # bottom edge = real axis: only the +θ leading zero, no conjugate
           xlabel=r"$\mathrm{Re}\,z/z_c$", ylabel=r"$\mathrm{Im}\,z/z_c$")
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
TSTAR7 = ("0.75", "1.15", "1.20")                   # columns: T* = 0.75, 1.15, 1.20
for i, (ax, t) in enumerate(zip(axs.flat, "abcdef")):
    panel(ax, t, right=t in "def", extra=rf"$T^*={TSTAR7[i % 3]}$")
save(fig, "fig7_lee_expansion")

## Fig. 8 — reach of the circle, term sizes, density from Lee's zeros (text width)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(TEXTW, 4.8))
a, c, b, d = axs.flat   # density panels stacked in the right column (bar chart bottom left)
lam, Tred = g("fig8/lam"), g("fig8/Tred"); kobs, kp, kx = g("fig8/kobs"), g("fig8/kpred"), g("fig8/kx")
ok = np.isfinite(kobs); X = (kp + 0.5) / (kx + 0.5); Y = (kobs + 0.5) / (kp + 0.5)
norm = mpl.colors.Normalize(Tred.min(), Tred.max()); cmap = TGRAD
SYSM = {0.0: ("o", "LJ"), 1.5: ("D", r"SW, $\lambda=1.50$"), 1.75: ("^", r"SW, $\lambda=1.75$")}
a.axvline(1, color=INK2, ls="--", lw=0.8); a.axhline(1, color=INK2, lw=0.8)
for L_, (m, lab) in SYSM.items():
    sel = ok & (lam == L_)
    a.scatter(X[sel], Y[sel], c=Tred[sel], cmap=cmap, norm=norm, marker=m, s=22, edgecolor=INK2, lw=0.5)
a.legend(handles=[Line2D([], [], ls="", marker=m, mfc="white", mec=INK2, ms=4.5, label=lab) for m, lab in SYSM.values()],
         loc="lower left")
a.set(xscale="log", xlim=(0.08, 25), ylim=(0.6, 1.5))
a.set_xlabel(r"$(\varepsilon_1+1/2)/(\varepsilon_\times+1/2)$"); a.set_ylabel(r"$(\varepsilon_1^{\rm obs}+1/2)/(\varepsilon_1+1/2)$", labelpad=2)
cax = a.inset_axes([1.015, 0.0, 0.045, 1.0])   # colour bar hugging the right edge of (a), a sliver of white between
cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax); cb.set_label(r"$T/T_c$")
Tb, t2, t3 = g("fig8/Tb"), g("fig8/t2"), g("fig8/t3"); f3 = t3 / (t2 + t3)
b.bar(Tb, 1 - f3, width=0.035, color=ORANGE, label="2nd-order term")
b.bar(Tb, f3, width=0.035, bottom=1 - f3, color=MAGENTA, label="3rd-order term")
b.axhline(0.5, color=INK2, ls="--", lw=0.8)
b.set(xlabel=r"$T^*$", ylabel=rf"Share of correction at $k={int(g('fig8/kbar'))}$", ylim=(0, 1.15), yticks=[0, 0.25, 0.5, 0.75, 1],
      xticks=[0.8, 0.9, 1.0, 1.1, 1.2])
# combined size |2nd| + |3rd| at k, in zero spacings, on a right-hand log axis
b.tick_params(axis="y", which="both", right=False)   # right side belongs to the twin axis
bt = b.twinx()
# |Δz|/z_c of the k-th zero ≈ |Δ ln z| = (size in zero spacings) × 2π/ΔN; ΔN per isotherm from the Julia run
_nm = np.genfromtxt("cache/numbers_for_manuscript.csv", delimiter=",", names=True)
_dN = np.interp(Tb, _nm["T_star"], _nm["DeltaN"])
bt.plot(Tb, (t2 + t3) * 2 * np.pi / _dN, "o-", color=REF, ms=3.5, lw=1.0, label=rf"Combined $|\Delta z_{{{int(g('fig8/kbar'))}}}|/z_c$ (right)")
bt.set_yscale("log"); bt.set_ylim(5e-4, 0.1)   # top at 10^-1; bottom just below the lowest point, so the curve clears the legend
bt.set_ylabel(rf"Combined displacement $|\Delta z_{{{int(g('fig8/kbar'))}}}|/z_c$", fontsize=9)
bt.tick_params(axis="y", which="both", direction="in")
h, l = b.get_legend_handles_labels(); h2, l2 = bt.get_legend_handles_labels()
b.legend(h[::-1] + h2, l[::-1] + l2, loc="center left", bbox_to_anchor=(0, 0.56))   # raised to clear the right-axis line by a sliver
b.minorticks_off(); b.tick_params(axis="x", which="both", top=False)
t = g("fig8/t")
for ax, T in ((c, "87.79"), (d, "140.46")):
    k = f"fig8/T{T}/"
    ax.plot(t, g(k + "rho"), color="black", lw=1.8, label=r"From $\ln Q$")
    for n, col, ls in ((1, BLUE, (0, (1, 0.8))), (2, ORANGE, "--"), (3, MAGENTA, "-.")):   # order 1: dense dots, colour readable
        ax.plot(t, g(k + f"rho{n}"), color=col, ls=ls, lw=1.8 if n == 1 else 1.3, label=f"Order {n}")
    ax.set(xlabel=r"$\beta\mu-\beta\mu_c$", ylabel=r"$\rho\;(\sigma^{-3})$")
    ax.legend(loc="upper left", bbox_to_anchor=(0.12, 0.99))
for ax, tg in zip(axs.flat, "abcd"): panel(ax, tg)
save(fig, "fig8_series_breakdown")